# MLflow Binary Classification Tutorial

This notebook demonstrates how to track multiple machine learning experiments using MLflow for a binary classification problem with imbalanced data.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

## Step 1: Import Required Libraries

First, we import all the necessary Python libraries:
- **numpy**: For numerical operations and array handling
- **sklearn**: Provides machine learning algorithms and tools
- **xgboost**: Advanced gradient boosting algorithm
- **warnings**: To suppress warning messages for cleaner output

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100], dtype=int64))

## Step 2: Create a Synthetic Dataset

Here we generate an artificial dataset for binary classification:
- **n_samples=1000**: Creates 1000 data points
- **n_features=10**: Each data point has 10 features
- **weights=[0.9, 0.1]**: Creates imbalanced data (90% class 0, 10% class 1)
- This simulates real-world scenarios like fraud detection where most cases are normal

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## Step 3: Split Data into Training and Testing Sets

We divide our dataset into two parts:
- **Training set (70%)**: Used to train the model
- **Testing set (30%)**: Used to evaluate model performance
- **stratify=y**: Ensures both sets maintain the same class ratio as the original data

## Experiment 1: Train Logistic Regression Classifier

**What is Logistic Regression?**
- A simple linear algorithm that predicts probabilities
- Good starting point for binary classification
- Parameters: C=1 (regularization), solver='liblinear' (optimization method)

In [4]:
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



## Experiment 2: Train Random Forest Classifier

**What is Random Forest?**
- An ensemble method that combines multiple decision trees
- More powerful than single models
- Parameters: n_estimators=30 (number of trees), max_depth=3 (how deep each tree grows)

In [5]:
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.96      0.99      0.98       270
           1       0.91      0.67      0.77        30

    accuracy                           0.96       300
   macro avg       0.94      0.83      0.87       300
weighted avg       0.96      0.96      0.96       300



## Experiment 3: Train XGBoost

**What is XGBoost?**
- An advanced gradient boosting algorithm
- Often provides the best performance
- Builds trees sequentially, each fixing errors from the previous one

In [6]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



## Experiment 4: Handle Class Imbalance with SMOTE + Train XGBoost

**Why do we need SMOTE?**
- Our data is imbalanced (90% class 0, 10% class 1)
- Models can be biased toward the majority class
- **SMOTE** (Synthetic Minority Over-sampling Technique) creates synthetic examples of the minority class
- This balances the dataset and improves model performance on rare cases

In [7]:
#pip install imbalanced-learn
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619], dtype=int64))

In [8]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



## Track Experiments Using MLflow

**What is MLflow?**
- A platform for tracking machine learning experiments
- Automatically logs parameters, metrics, and models
- Helps compare different models and choose the best one
- Provides a web interface to visualize all experiments

In [9]:
models = [
    (
        "Logistic Regression", 
        LogisticRegression(C=1, solver='liblinear'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        RandomForestClassifier(n_estimators=30, max_depth=3), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

## Step 4: Define All Models to Compare

We create a list of all models we want to test:
- Each entry contains: model name, parameters, model object, training data, and test data
- This allows us to train and compare all models systematically

In [10]:
reports = []

for model_name, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

## Step 5: Train All Models and Generate Reports

This loop:
1. Takes each model from our list
2. Trains it on the training data
3. Makes predictions on test data
4. Generates a detailed performance report
5. Stores all reports for MLflow tracking

In [11]:
import mlflow

## Step 6: Import MLflow Libraries

We import MLflow modules for:
- **mlflow**: Core tracking functionality
- **mlflow.sklearn**: For logging scikit-learn models
- **mlflow.xgboost**: For logging XGBoost models

In [13]:
# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
mlflow.set_tracking_uri("http://127.0.0.1:5000")

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_param("model", model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2026/01/26 09:40:56 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection' does not exist. Creating a new experiment.


2026/01/26 09:40:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/26 09:41:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/2/runs/1cdad4be11174a18824a7e62153731e5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/01/26 09:41:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/2/runs/83daf3fe5f224434a16322d1ceabef42
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/01/26 09:41:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/2/runs/0d834c16819d43b89ecfbc38189cfa92
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/2/runs/b082eab1611440e49c6bfd67e761ab98
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


## Step 7: Track All Experiments in MLflow

This is where the magic happens! For each model:
1. **Set experiment name**: Groups related runs together
2. **Set tracking URI**: Points to MLflow server (local or remote)
3. **Start a run**: Creates a new experiment record
4. **Log parameters**: Records model settings (e.g., learning_rate, max_depth)
5. **Log metrics**: Records performance (accuracy, recall, f1-score)
6. **Log model**: Saves the trained model for future use

After running this, open http://127.0.0.1:5000 to see all experiments!